# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [19]:
!pip install langchain_community langchain_experimental langchain_openai
!pip install faiss-cpu google-search-results


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [20]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [21]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [22]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [23]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [24]:
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

In [25]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [26]:
OBJECTIVE = "Write a weather report for SF today"

In [27]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [28]:
baby_agi({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Question: What is the weather report for San Francisco today?
Thought: I need to find the current weather conditions for San Francisco to provide an accurate report.
Action: Search
Action Input: "current weather San Francisco October 2023"
Observation: ["Find out what kind of weather we're expecting this weekend. Thursday afternoon forecast - Oct. 5, 2023. Northern California weather forecast - ...", 'Light rainfall is possible across the San Francisco Bay Area on Monday and Tuesday, with accumulations of less than a quarter inch.', 'October Day +82° °. Average values in October Wind speed 15.4 mph Pressure 30 inHg Humidity 57 % Average daytime values for October 2020.', "October is one of San Francisco's warmest and driest months, with average highs around 70°F and lows near 55°F. rainfall is light—about 1.2 inches for the ...", 'San Francisco, CA warning 57 °FClear. Record High Temp 68 71.1 90 Low Temp 56 52.5 40 Day Average Temp 60.39 61.8 - Precipitation (in)', 'KRON4 weather ancho

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



*****TASK LIST*****

2: Write a weather report for San Francisco for October 6, 2023
3: Provide a comparison of today's weather in San Francisco with the average October weather
4: Summarize the weather trends in San Francisco for the week of October 1-7, 2023
5: Create a list of outdoor activities suitable for mild weather in San Francisco
6: Research and report on the historical weather patterns in San Francisco during October

*****NEXT TASK*****

2: Write a weather report for San Francisco for October 6, 2023
Thought: I need to find the current weather conditions for San Francisco on October 6, 2023, to create an accurate weather report.
Action: Search
Action Input: "San Francisco weather report October 6 2023"
Observation: ['It reports on an extreme heat risk in Southern California, the Bay Area, and the Central Valley, with a potential for a tropical depression ...', 'This report shows the past weather for San Francisco, providing a weather history for October 2022. It features 

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



*****TASK LIST*****

3: Provide a comparison of today's weather in San Francisco with the average October weather.
4: Analyze the impact of current weather conditions on local events in San Francisco today.
5: Investigate and report on any weather-related news or alerts affecting San Francisco today.
6: Write a detailed forecast for San Francisco's weather for October 7, 2023.
7: Summarize the weather trends in San Francisco for the week of October 1-7, 2023.
8: Create a guide for residents on how to prepare for sudden weather changes in San Francisco during October.
9: Research and report on the historical weather patterns in San Francisco during October.
10: Create a list of outdoor activities suitable for mild weather in San Francisco.

*****NEXT TASK*****

3: Provide a comparison of today's weather in San Francisco with the average October weather.
Thought: I need to gather the current weather data for San Francisco today and compare it with the average weather for October. 
Actio

{'objective': 'Write a weather report for SF today'}